# Analysis: Political Violence in Myanmar After the 2021 Coup

This notebook analyzes how the intensity and geographic spread of political violence in Myanmar changed after the February 2021 military coup using ACLED data.

Each section will also have a brief explanation on the role of the code. 

## Load Cleaned Dataset

This section loads the cleaned data set from (01_data_cleaning.ipynb) to use for further analysis. 

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio
import folium

pio.templates.default = "plotly_dark"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Code" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"

df = pd.read_csv(DATA_DIR / "cleaned_acled.csv")

df.head()

,event_id_cnty,event_date,year,time_precision,disorder_type,event_type,sub_event_type,actor1,assoc_actor_1,inter1,...,longitude,geo_precision,source,source_scale,notes,fatalities,tags,timestamp,period,month
0,MMR6789,2019-04-29,2019,1,Political violence,Battles,Armed clash,KNU/KNLA: Karen National Union/Karen National ...,NaN,Rebel group,...,98.1232,1,Irrawaddy,National,"On 29 April 2019, near the area of Three-Pagod...",0,NaN,1557838713,Before Coup,2019-04
1,MMR7145,2019-02-26,2019,1,Political violence,Violence Against Civilians,Attack,Private Security Forces (Myanmar),Labor Group (Myanmar),External/Other forces,...,97.4384,1,Irrawaddy,National,"On 26 February 2019, in Waingmaw Township, in ...",0,NaN,1559057912,Before Coup,2019-02
2,MMR6387,2019-01-08,2019,2,Political violence,Violence Against Civilians,Abduction/forced disappearance,PSLF/TNLA: Palaung State Liberation Front/Ta'a...,NaN,Rebel group,...,97.6798,2,Myanmar Times,National,"On 8 January 2019, in Namkham Township, in Sha...",0,NaN,1618564830,Before Coup,2019-01
3,MMR6478,2019-02-16,2019,2,Political violence,Violence Against Civilians,Attack,Unidentified Armed Group (Myanmar),NaN,Political militia,...,92.3661,2,Myanmar Times,National,"On 16 February 2019, near Taung Pyo Let Wae Vi...",3,NaN,1563908691,Before Coup,2019-02
4,MMR6242,2018-12-16,2018,1,Political violence,Violence Against Civilians,Attack,Unidentified Armed Group (Myanmar),NaN,Political militia,...,92.3661,2,Irrawaddy,National,"On 16 December 2018, in Maungdaw township, Rak...",1,NaN,1563908691,Before Coup,2018-12


## Research Question

How did the intensity and geographic spread of political violence in Myanmar change after the 2021 military coup?

## 1. Change in Event Frequency

Groups the dataset by month to count the number of events and converts the time variable into a usable datetime format.

Plots a line graph to show how event frequency changes over time.

Adds a vertical marker to indicate the February 2021 coup.

Applies a dark theme and formats the axis to show clean yearly intervals.

In [2]:
monthly_events = df.groupby('month').size().reset_index(name='Events')
monthly_events['month'] = pd.to_datetime(monthly_events['month'].astype(str))

fig = px.line(
    monthly_events,
    x='month',
    y='Events',
    title='Monthly Political Violence Events in Myanmar',
    labels={
        'month': 'Month',
        'Events': 'Number of Events'
    }
)

fig.update_traces(
    line=dict(color='#4cc9f0', width=3),
    hovertemplate="<b>%{x|%B %Y}</b><br>Events: %{y}<extra></extra>"
)

fig.update_traces(line=dict(color='#4cc9f0', width=3))

# Coup line
fig.add_shape(
    type="line",
    x0="2021-02-01",
    x1="2021-02-01",
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="white", dash="dash", width=2)
)

fig.add_annotation(
    x="2021-02-01",
    y=1,
    xref="x",
    yref="paper",
    text="Military Coup",
    showarrow=False,
    yshift=10,
    font=dict(color="white")
)

# KEY PART — restore clean axis look
fig.update_layout(
    template='plotly_dark',
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font=dict(family='Helvetica', color='white'),
    
    xaxis=dict(
        title='Month',
        showgrid=True,
        gridcolor='#333',
        tickformat='%Y',   # ← CLEAN yearly labels
        dtick="M12"        # ← ONE tick per year (fixes clutter)
    ),
    
    yaxis=dict(
        title='Number of Events',
        showgrid=True,
        gridcolor='#333'
    )
)

fig.show()

## 2. Fatalities

Aggregates total fatalities by month and converts the time variable into datetime format.

Plots a line graph to show how fatalities change over time.

Marks the February 2021 coup with a vertical line and lightly shades the post coup period.

Applies a dark theme and labels axes to clearly present trends in conflict lethality.

In [3]:
monthly_fatalities = df.groupby('month')['fatalities'].sum().reset_index(name='Fatalities')
monthly_fatalities['month'] = pd.to_datetime(monthly_fatalities['month'].astype(str))

fig = px.line(
    monthly_events,
    x='month',
    y='Events',
    title='Monthly Political Violence Events in Myanmar',
    labels={
        'month': 'Month',
        'Events': 'Number of Events'
    }
)

fig.update_traces(
    line=dict(color='#4cc9f0', width=3),
    hovertemplate="<b>%{x|%B %Y}</b><br>Events: %{y}<extra></extra>"
)

fig.update_traces(line=dict(color='#4cc9f0', width=3))

fig.update_traces(line=dict(color='#4cc9f0', width=3))

fig.add_shape(
    type="line",
    x0="2021-02-01",
    x1="2021-02-01",
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(color="white", dash="dash", width=2)
)

fig.add_annotation(
    x="2021-02-01",
    y=1,
    xref="x",
    yref="paper",
    text="Military Coup",
    showarrow=False,
    yshift=10,
    font=dict(color="white")
)

fig.add_vrect(
    x0="2021-02-01",
    x1=monthly_fatalities['month'].max(),
    fillcolor="#4cc9f0",
    opacity=0.08,
    layer="below",
    line_width=0
)

fig.update_layout(
    template='plotly_dark',
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font=dict(family='Helvetica', color='white'),
    xaxis_title='Month',
    yaxis_title='Number of Fatalities'
)

fig.show()

## 3. Types of Violence

Cleans and standardizes event type and period labels for consistency.

Groups data by period and event type to count the number of events in each category.

Creates a grouped bar chart to compare types of violence before and after the coup.

Applies a dark theme and custom colors to clearly highlight differences in conflict composition.

In [4]:
df['event_type'] = df['event_type'].replace({
    'Explosions/Remote violence': 'Explosions and Remote Attacks',
    'Explosions & Remote Attacks': 'Explosions and Remote Attacks',
    'Violence against civilians': 'Violence Against Civilians'
})

df['period'] = df['period'].replace({
    'pre_coup': 'Before Coup',
    'post_coup': 'After Coup'
})

event_types = df.groupby(['period', 'event_type']).size().reset_index(name='Events')

fig = px.bar(
    event_types,
    x='period',
    y='Events',
    color='event_type',
    barmode='group',
    title='Types of Political Violence Before and After the Coup',
    labels={
        'period': '',
        'event_type': 'Type of Violence',
        'Events': 'Number of Events'
    },
    color_discrete_sequence=['#4cc9f0', '#4895ef', '#4361ee']
)

fig.update_traces(
    hovertemplate='<b>%{x}</b><br>%{fullData.name}: %{y}<extra></extra>'
)

fig.update_layout(
    template='plotly_dark',
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font=dict(family='Helvetica', color='white'),
    legend_title_text='Type of Violence',
    xaxis_title='',
    yaxis_title='Number of Events',
    xaxis=dict(
        categoryorder='array',
        categoryarray=['Before Coup', 'After Coup']
    )
)

fig.show()

## 4. Geographic Spread

Generates an interactive map centered on Myanmar using a light basemap for clearer visibility.

Samples event data and plots each incident as a colored marker, distinguishing between pre and post coup periods.

Uses color coding to visually show how conflict spreads geographically over time.

Outputs an interactive map that allows exploration of spatial patterns in political violence.

In [5]:
import folium

m = folium.Map(
    location=[21.9, 96.0],
    zoom_start=5,
    tiles="CartoDB Voyager"
)

for _, row in df.sample(2000).iterrows():
    color = '#4cc9f0' if row['period'] == 'Before Coup' else '#f72585'
    
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=2,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6
    ).add_to(m)

df['period'].unique()

m

## Average Fatalities per Event

Groups the data by period to calculate the average number of fatalities per event before and after the coup.  

Creates a bar chart to compare lethality across the two periods.  

Applies consistent styling and formatting to match the overall visual theme.  

Shows how individual events became more or less deadly on average.

In [6]:
avg_fatalities = df.groupby('period')['fatalities'].mean().reset_index(name='Average Fatalities')

fig = px.bar(
    avg_fatalities,
    x='period',
    y='Average Fatalities',
    title='Average Fatalities per Event Before and After the Coup',
    color='period',
    color_discrete_sequence=['#4895ef', '#4cc9f0']
)

fig.update_traces(
    hovertemplate='<b>%{x}</b><br>Average fatalities per event: %{y:.2f}<extra></extra>'
)

fig.update_layout(
    template='plotly_dark',
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    font=dict(family='Helvetica', color='white'),
    xaxis_title='',
    yaxis_title='Average Fatalities per Event',
    showlegend=False,
    xaxis=dict(
        categoryorder='array',
        categoryarray=['Before Coup', 'After Coup']
    )
)

fig.show()